In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, DoubleType, BooleanType, DateType

In [0]:
spark.conf.set("fs.azure.account.auth.type.dataofolympics.dfs.core.windows.net", "OAuth")

spark.conf.set("fs.azure.account.oauth.provider.type.dataofolympics.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")

spark.conf.set("fs.azure.account.oauth2.client.id.dataofolympics.dfs.core.windows.net", "<client-id>")

spark.conf.set("fs.azure.account.oauth2.client.secret.dataofolympics.dfs.core.windows.net", "<client-secret>")

spark.conf.set("fs.azure.account.oauth2.client.endpoint.dataofolympics.dfs.core.windows.net",
               "https://login.microsoftonline.com/<tenant-id>/oauth2/token")

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .load("abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/raw_data/athletes.csv")

In [0]:
display(
  dbutils.fs.ls("abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/raw_data/")
)

path,name,size,modificationTime
abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/raw_data/athletes.csv,athletes.csv,418482,1775389882000
abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/raw_data/coaches.csv,coaches.csv,16885,1775389899000
abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/raw_data/gender.csv,gender.csv,1123,1775389916000
abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/raw_data/medals.csv,medals.csv,2410,1775389933000
abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/raw_data/teams.csv,teams.csv,35262,1775389950000


In [0]:
athletes = spark.read.format("csv").option('header','true').load("abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/raw_data/athletes.csv")
coaches = spark.read.format("csv").option('header','true').load("abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/raw_data/coaches.csv")
gender = spark.read.format("csv").option('header','true').option("inferSchema","true").load("abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/raw_data/gender.csv")
medals = spark.read.format("csv").option('header','true').option("inferSchema","true").load("abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/raw_data/medals.csv")
teams = spark.read.format("csv").option('header','true').load("abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/raw_data/teams.csv")

In [0]:
print('athletes')
athletes.show(n=5)  
athletes.printSchema()
print('coaches')
coaches.show(n=5)
coaches.printSchema()
print('gender')
gender.show(n=5)
gender.printSchema()
print('medals')
medals.show(n=5)
medals.printSchema()
print('teams')
teams.show(n=5)
teams.printSchema()

athletes
+-----------------+------+-------------------+
|             Name|   NOC|         Discipline|
+-----------------+------+-------------------+
|  AALERUD Katrine|Norway|       Cycling Road|
|      ABAD Nestor| Spain|Artistic Gymnastics|
|ABAGNALE Giovanni| Italy|             Rowing|
|   ABALDE Alberto| Spain|         Basketball|
|    ABALDE Tamara| Spain|         Basketball|
+-----------------+------+-------------------+
only showing top 5 rows

root
 |-- Name: string (nullable = true)
 |-- NOC: string (nullable = true)
 |-- Discipline: string (nullable = true)

coaches
+---------------+-------------+----------+-----+
|           Name|          NOC|Discipline|Event|
+---------------+-------------+----------+-----+
|ABDELMAGID Wael|        Egypt|  Football| NULL|
|      ABE Junya|        Japan|Volleyball| NULL|
|  ABE Katsuhiko|        Japan|Basketball| NULL|
|   ADAMA Cherif|C�te d'Ivoire|  Football| NULL|
|     AGEBA Yuya|        Japan|Volleyball| NULL|
+---------------+-------

In [0]:
gender = gender.withColumn('Female', col('Female').cast(IntegerType()))\
        .withColumn('Male', col('Male').cast(IntegerType()))\
        .withColumn('Total', col('Total').cast(IntegerType()))




In [0]:
top_gold_medalist_countries = medals.orderBy("Gold", ascending = False).select("Rank","Team/NOC","Gold").show()

+----+--------------------+----+
|Rank|            Team/NOC|Gold|
+----+--------------------+----+
|   1|United States of ...|  39|
|   2|People's Republic...|  38|
|   3|               Japan|  27|
|   4|       Great Britain|  22|
|   5|                 ROC|  20|
|   6|           Australia|  17|
|   7|         Netherlands|  10|
|   8|              France|  10|
|   9|             Germany|  10|
|  10|               Italy|  10|
|  11|              Canada|   7|
|  12|              Brazil|   7|
|  13|         New Zealand|   7|
|  14|                Cuba|   7|
|  15|             Hungary|   6|
|  16|   Republic of Korea|   6|
|  17|              Poland|   4|
|  18|      Czech Republic|   4|
|  19|               Kenya|   4|
|  20|              Norway|   4|
+----+--------------------+----+
only showing top 20 rows



In [0]:
average_entries_by_gender = gender.withColumn(
    'Avg_female', col('Female') / col('Total')
).withColumn(
    'Avg_male', col('Male') / col('Total')
)

average_entries_by_gender.show()

+--------------------+------+----+-----+-------------------+-------------------+
|          Discipline|Female|Male|Total|         Avg_female|           Avg_male|
+--------------------+------+----+-----+-------------------+-------------------+
|      3x3 Basketball|    32|  32|   64|                0.5|                0.5|
|             Archery|    64|  64|  128|                0.5|                0.5|
| Artistic Gymnastics|    98|  98|  196|                0.5|                0.5|
|   Artistic Swimming|   105|   0|  105|                1.0|                0.0|
|           Athletics|   969|1072| 2041| 0.4747672709456149| 0.5252327290543851|
|           Badminton|    86|  87|  173|0.49710982658959535| 0.5028901734104047|
|   Baseball/Softball|    90| 144|  234|0.38461538461538464| 0.6153846153846154|
|          Basketball|   144| 144|  288|                0.5|                0.5|
|    Beach Volleyball|    48|  48|   96|                0.5|                0.5|
|              Boxing|   102

In [0]:
athletes.write.option("header",'true').mode("overwrite").csv("abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/transformed_data/athletes")
coaches.write.option("header",'true').mode("overwrite").csv("abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/transformed_data/coaches")
gender.write.option("header",'true').mode("overwrite").csv("abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/transformed_data/gender")
medals.write.option("header",'true').mode("overwrite").csv("abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/transformed_data/medals")
teams.write.option("header",'true').mode("overwrite").csv("abfss://olympics-data-container@dataofolympics.dfs.core.windows.net/transformed_data/teams")
